# Day 42 — Practice Questions

## Naive Bayes

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | Counting and Bayes' rule | 1 – 4 |
| **B** | The naive assumption | 5 – 7 |
| **C** | Smoothing and logs | 8 – 10 |
| **D** | Choosing a flavour | 11 – 12 |

Run the setup cell first. Every question is self-contained after that.

> Replace every `...` with your own code. Where a question asks you to *explain*,
> write two or three sentences in the markdown cell — the explanation is the point.

In [ ]:
import math, time, warnings
import numpy as np
warnings.simplefilter("ignore")

from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.datasets import load_digits, load_breast_cancer, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

CV = StratifiedKFold(5, shuffle=True, random_state=0)

# ---- a NEW corpus, different from the one in the teaching notebook ---------
# 24 product reviews. 1 = a complaint, 0 = a compliment.
BAD = [
    "the battery died after one week",
    "screen cracked on day two",
    "stopped working after a month",
    "the battery is terrible",
    "broke immediately do not buy",
    "poor quality and slow delivery",
    "screen is dim and the battery drains",
    "it stopped charging completely",
    "cheap plastic feels fragile",
    "returned it the same week",
]
GOOD = [
    "excellent battery life so far",
    "fast delivery and well packed",
    "works perfectly every day",
    "great screen and sharp colours",
    "lovely build quality",
    "exactly what i wanted",
    "the battery lasts all day",
    "arrived early and works well",
    "very happy with this",
    "sturdy and reliable so far",
    "clear screen and easy setup",
    "good value for the price",
    "comfortable and light",
    "does the job nicely",
]

REVIEWS = [(r, 1) for r in BAD] + [(r, 0) for r in GOOD]   # 1 = complaint

def has(doc, word):
    return word in doc.split()

def count(word, label=None):
    return sum(1 for d, y in REVIEWS if has(d, word) and (label is None or y == label))

def n(label=None):
    return sum(1 for _, y in REVIEWS if label is None or y == label)

print("reviews:", n(), " complaints:", n(1), " compliments:", n(0))
for w in ["battery", "screen", "works"]:
    print(f'  "{w}" appears in {count(w)} reviews '
          f'({count(w,1)} complaint, {count(w,0)} compliment)')

---
# Part A · Counting and Bayes' rule

## Question 1

A new review contains the word **battery**. Work out `P(complaint | battery)` the way
we did in class — *without any formula*. Print every review containing "battery", then
the fraction of those that are complaints.

In [ ]:
survivors = [(d, y) for d, y in REVIEWS if ...]

for d, y in survivors:
    print(f"   {'COMPLAINT ' if y else 'compliment'}   {d}")

print()
print("P(complaint | battery) =", ...)

## Question 2

Now get the same number out of Bayes' rule. Compute the prior, the likelihood and the
evidence separately, combine them, and check the result against Question 1.

$$P(\text{complaint} \mid \text{battery}) =
\frac{P(\text{battery} \mid \text{complaint}) \times P(\text{complaint})}{P(\text{battery})}$$

In [ ]:
prior      = ...                  # P(complaint)
likelihood = ...                  # P(battery | complaint)
evidence   = ...                  # P(battery)

posterior = ...

print("prior      ", round(prior, 4))
print("likelihood ", round(likelihood, 4))
print("evidence   ", round(evidence, 4))
print("posterior  ", round(posterior, 4))

## Question 3

Here are two statements about this corpus:

- **(i)** `P(battery | complaint)` = 0.30
- **(ii)** `P(complaint | battery)` = 0.60

Both are true and they are different numbers. Explain in your own words what each one
is actually asking, and describe a situation where confusing the two would lead
somebody to a badly wrong conclusion.

*Your answer:*

## Question 4

The word **screen** appears in both classes. Work out `P(complaint | screen)` and
compare it with the prior `P(complaint)`.

Then answer: is "screen" a useful clue? What would make a clue useless, and what would
make one perfect?

In [ ]:
for w in ["battery", "screen"]:
    p = ...
    print(f"{w:>9}  appears {count(w):>2} times   "
          f"P(complaint | {w}) = {p:.4f}")

print("prior P(complaint) =", round(n(1)/n(), 4))

---
# Part B · The naive assumption

## Question 5

Count how many **separate probabilities** you would have to estimate to do the honest
(non-naive) calculation, as the number of words you look at grows.

Print a table for 1, 2, 3, 5, 10 and 20 words, and compare each to the 24 reviews you
actually have.

In [ ]:
print(f"{'words':>6}  {'combinations':>14}  {'reviews per combination':>24}")
for k in [1, 2, 3, 5, 10, 20]:
    combos = ...
    print(f"{k:>6}  {combos:>14,}  {n()/combos:>24.6f}")

## Question 6

Measure how wrong the naive assumption is on **this** corpus, for the words
**battery** and **screen**, within the complaints.

Compare the exact joint probability with the product of the two separate ones.

In [ ]:
def joint(w1, w2, label):
    return sum(1 for d, y in REVIEWS if y == label and has(d, w1) and has(d, w2))

exact = ...
naive = ...

print("exact P(battery and screen | complaint) =", round(exact, 4))
print("naive P(battery|c) x P(screen|c)        =", round(naive, 4))
print("ratio                                   =", round(exact / naive, 3))

## Question 7

The assumption is false — you just measured it, twice, on two different corpora. Yet
naive Bayes remains one of the most widely deployed classifiers ever written.

Explain why being wrong about the probabilities does not necessarily make it wrong
about the **labels**. Then say what this implies about using `predict_proba()`.

*Your answer:*

---
# Part C · Smoothing and logs

## Question 8

Classify the review **"the battery works"** with **no smoothing**, by hand.

Write a `score` function that multiplies the prior by one likelihood per word, run it
for both classes, and print the resulting probability of "complaint".

Something will go wrong. Say what, and why.

In [ ]:
def score(words, label, alpha=0.0):
    p = n(label) / n()
    for w in words:
        p *= ...
    return p

words = ["the", "battery", "works"]
s_bad, s_good = score(words, 1), score(words, 0)

print("complaint score  ", s_bad)
print("compliment score ", s_good)
print("P(complaint)     ", ...)

## Question 9

Repair it with Laplace smoothing. Sweep `alpha` over
`[0, 0.01, 0.1, 0.5, 1, 5, 20]` and print, for each value, the smoothed
`P(works | complaint)`, both scores, and the resulting `P(complaint)`.

Then say what happens at the two extremes of the sweep.

In [ ]:
print(f"{'alpha':>7}  {'P(works|comp)':>14}  {'P(complaint)':>13}  verdict")
for a in [0, 0.01, 0.1, 0.5, 1, 5, 20]:
    sb, sg = score(words, 1, a), score(words, 0, a)
    pw = ...
    print(f"{a:>7}  {pw:>14.4f}  {sb/(sb+sg):>13.4f}  "
          f"{'complaint' if sb > sg else 'compliment'}")

## Question 10

Show the underflow problem for yourself, then show that logs fix it.

Multiply 0.01 by itself 150, 200 and 400 times and print each result. Then print the
sum of logs for the same cases. Explain why swapping to logs is allowed — what property
makes it safe?

In [ ]:
for k in [150, 200, 400]:
    product = ...
    logsum  = ...
    print(f"{k:>4} clues   product {product:>12.3e}   log-sum {logsum:>10.3f}")

---
# Part D · Choosing a flavour

## Question 11

Three datasets, three flavours. For each dataset below, run **all three** naive Bayes
flavours plus a random forest, and print the cross-validated accuracy.

- `load_digits()` — 64 columns of pixel **counts** (0–16)
- `load_breast_cancer()` — 30 columns of continuous **measurements**
- `load_wine()` — 13 columns of continuous **measurements** on very different scales

Then say which flavour won each one, and whether that is what the slides predicted.

In [ ]:
sets = {"digits": load_digits(return_X_y=True),
        "breast_cancer": load_breast_cancer(return_X_y=True),
        "wine": load_wine(return_X_y=True)}

for name, (X, y) in sets.items():
    print(f"\n{name}   ({X.shape[0]} rows x {X.shape[1]} columns)")
    for label, model in [("MultinomialNB", ...), ("GaussianNB", ...),
                         ("BernoulliNB", ...), ("random forest", ...)]:
        print(f"   {label:<16} {cross_val_score(model, X, y, cv=CV).mean():.4f}")

## Question 12

For each situation, say whether you would reach for **naive Bayes** or something else,
and give one sentence of justification.

1. 40,000 support emails to route to the right department, retrained every night.
2. An insurer needs the probability a claim is fraudulent, to decide how much to set aside.
3. A first look at a brand-new dataset at 4pm, with a meeting at 4:30.
4. 200 patients, 20,000 gene columns, published accuracy is what matters.
5. Predicting house price from square footage and number of bedrooms — two columns that move together closely.

*Your answers:*

1.
2.
3.
4.
5.

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload
the notebook. Questions 3, 7, 10 and 12 are marked mostly on the explanation — a correct
number with no reasoning gets little credit.

Solutions will be released with Day 43.